# Analyse Exploratoire du LIAR Dataset
## Detection automatique de fake news politiques

**Objectif** : Explorer et comprendre le LIAR Dataset (~12 800 declarations politiques) avant la modelisation.

**Plan** :
1. Chargement et structure des donnees
2. Analyse des labels de veracite (6 classes + mapping binaire)
3. Analyse textuelle (longueur, vocabulaire)
4. Analyse des metadonnees (speakers, partis, sujets)
5. Preprocessing et sauvegarde des splits

**Source** : Wang, W.Y. (2017). "Liar, Liar Pants on Fire": A New Benchmark Dataset for Fake News Detection. ACL 2017.

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path
from collections import Counter
import re
import warnings
warnings.filterwarnings("ignore")

# Chemins
DATA_RAW = Path("../data/brutes/liar_dataset")
DATA_MERGED = Path("../data/fusionnes")
DATA_OUT = Path("../data/Traitees")
DATA_OUT.mkdir(parents=True, exist_ok=True)

# Palette de couleurs cohérente
COLORS_6 = ["#e74c3c", "#e67e22", "#f39c12", "#3498db", "#2ecc71", "#27ae60"]
COLORS_BIN = ["#e74c3c", "#2ecc71"]

print("Imports OK")

Imports OK


## 1. Chargement des donnees

Le LIAR dataset est fourni en 3 splits (train/valid/test) au format TSV sans en-tete.
Les colonnes sont : `id, label, statement, subject, speaker, job_title, state_info, party, barely_true_counts, false_counts, half_true_counts, mostly_true_counts, pants_on_fire_counts, context`.

In [2]:
COLUMNS = [
    "id", "label", "statement", "subject", "speaker",
    "job_title", "state_info", "party",
    "barely_true_counts", "false_counts", "half_true_counts",
    "mostly_true_counts", "pants_on_fire_counts", "context"
]

# Chargement des 3 splits
df_train = pd.read_csv(DATA_RAW / "train.tsv", sep="\t", header=None, names=COLUMNS)
df_valid = pd.read_csv(DATA_RAW / "valid.tsv", sep="\t", header=None, names=COLUMNS)
df_test  = pd.read_csv(DATA_RAW / "test.tsv",  sep="\t", header=None, names=COLUMNS)

# Ajout de la colonne split
df_train["split"] = "train"
df_valid["split"] = "valid"
df_test["split"]  = "test"

# Fusion
df = pd.concat([df_train, df_valid, df_test], ignore_index=True)

print(f"Taille totale : {len(df):,} declarations")
print(f"  Train : {len(df_train):,}")
print(f"  Valid : {len(df_valid):,}")
print(f"  Test  : {len(df_test):,}")
print(f"\nColonnes : {list(df.columns)}")
df.head(3)

Taille totale : 12,791 declarations
  Train : 10,240
  Valid : 1,284
  Test  : 1,267

Colonnes : ['id', 'label', 'statement', 'subject', 'speaker', 'job_title', 'state_info', 'party', 'barely_true_counts', 'false_counts', 'half_true_counts', 'mostly_true_counts', 'pants_on_fire_counts', 'context', 'split']


,id,label,statement,subject,speaker,job_title,state_info,party,barely_true_counts,false_counts,half_true_counts,mostly_true_counts,pants_on_fire_counts,context,split
0,2635.json,false,Says the Annies List political group supports ...,abortion,dwayne-bohac,State representative,Texas,republican,0.0,1.0,0.0,0.0,0.0,a mailer,train
1,10540.json,half-true,When did the decline of coal start? It started...,"energy,history,job-accomplishments",scott-surovell,State delegate,Virginia,democrat,0.0,0.0,1.0,1.0,0.0,a floor speech.,train
2,324.json,mostly-true,"Hillary Clinton agrees with John McCain ""by vo...",foreign-policy,barack-obama,President,Illinois,democrat,70.0,71.0,160.0,163.0,9.0,Denver,train


In [3]:
# Apercu des types et valeurs manquantes
print("=== Types ===")
print(df.dtypes)
print(f"\n=== Valeurs manquantes ===")
missing = df.isnull().sum()
print(missing[missing > 0])
print(f"\nTotal lignes avec au moins un NaN : {df.isnull().any(axis=1).sum()}")

=== Types ===
id                       object
label                    object
statement                object
subject                  object
speaker                  object
job_title                object
state_info               object
party                    object
barely_true_counts      float64
false_counts            float64
half_true_counts        float64
mostly_true_counts      float64
pants_on_fire_counts    float64
context                  object
split                    object
dtype: object

=== Valeurs manquantes ===
subject                    2
speaker                    2
job_title               3568
state_info              2751
party                      2
barely_true_counts         2
false_counts               2
half_true_counts           2
mostly_true_counts         2
pants_on_fire_counts       2
context                  131
dtype: int64

Total lignes avec au moins un NaN : 4356


## 2. Analyse des labels de veracite

Le LIAR dataset comporte **6 niveaux de veracite** : `pants-fire`, `false`, `barely-true`, `half-true`, `mostly-true`, `true`.

Pour la classification, on cree plusieurs mappings :
- **Binaire** (Fake=0, Real=1) : le plus simple, bon pour la generalisation
- **3 classes** (Faux=0, Ambigu=1, Vrai=2) : bon compromis pedagogique
- **6 classes** : le plus fin, mais le desequilibre rend la tache tres difficile

In [4]:
# Ordre naturel des labels (du plus faux au plus vrai)
LABEL_ORDER = ["pants-fire", "false", "barely-true", "half-true", "mostly-true", "true"]

# Mapping binaire : Fake (0) vs Real (1)
BINARY_MAP = {
    "pants-fire": 0, "false": 0, "barely-true": 0,
    "half-true": 1, "mostly-true": 1, "true": 1,
}
BINARY_NAMES = {0: "Fake", 1: "Real"}

# Mapping 3 classes : Faux (0), Ambigu (1), Vrai (2)
TERNARY_MAP = {
    "pants-fire": 0, "false": 0,
    "barely-true": 1, "half-true": 1,
    "mostly-true": 2, "true": 2,
}
TERNARY_NAMES = {0: "Faux", 1: "Ambigu", 2: "Vrai"}

df["label_binary"] = df["label"].map(BINARY_MAP)
df["label_binary_name"] = df["label_binary"].map(BINARY_NAMES)
df["label_3class"] = df["label"].map(TERNARY_MAP)
df["label_3class_name"] = df["label_3class"].map(TERNARY_NAMES)

# Distribution des 6 classes
label_counts = df["label"].value_counts().reindex(LABEL_ORDER)
print("Distribution des 6 classes :")
print(label_counts)
print(f"\nRatio min/max : {label_counts.min() / label_counts.max():.2f}")
print(f"\nDistribution 3 classes : {df['label_3class_name'].value_counts().to_dict()}")
print(f"Distribution binaire   : {df['label_binary_name'].value_counts().to_dict()}")

Distribution des 6 classes :
label
pants-fire     1047
false          2507
barely-true    2103
half-true      2627
mostly-true    2454
true           2053
Name: count, dtype: int64

Ratio min/max : 0.40

Distribution 3 classes : {'Ambigu': 4730, 'Vrai': 4507, 'Faux': 3554}
Distribution binaire   : {'Real': 7134, 'Fake': 5657}


In [5]:
# Visualisation : Distribution des 6 classes
fig = px.bar(
    x=LABEL_ORDER,
    y=label_counts.values,
    color=LABEL_ORDER,
    color_discrete_sequence=COLORS_6,
    labels={"x": "Label de veracite", "y": "Nombre de declarations"},
    title="Distribution des 6 classes de veracite (LIAR Dataset)",
    text=label_counts.values,
)
fig.update_traces(textposition="outside")
fig.update_layout(showlegend=False, template="plotly_white", height=450)
fig.show()

In [6]:
# Visualisation : Distribution binaire (Fake vs Real)
bin_counts = df["label_binary_name"].value_counts().reindex(["Fake", "Real"])

fig = px.pie(
    names=bin_counts.index,
    values=bin_counts.values,
    color=bin_counts.index,
    color_discrete_map={"Fake": "#e74c3c", "Real": "#2ecc71"},
    title="Distribution binaire : Fake vs Real",
    hole=0.4,
)
fig.update_traces(textinfo="label+percent+value", textfont_size=14)
fig.update_layout(template="plotly_white", height=400)
fig.show()

print(f"Fake : {bin_counts['Fake']:,} ({bin_counts['Fake']/len(df)*100:.1f}%)")
print(f"Real : {bin_counts['Real']:,} ({bin_counts['Real']/len(df)*100:.1f}%)")

Fake : 5,657 (44.2%)
Real : 7,134 (55.8%)


In [7]:
# Distribution des labels par split (train/valid/test)
split_label = df.groupby(["split", "label"]).size().reset_index(name="count")

fig = px.bar(
    split_label, x="label", y="count", color="split",
    barmode="group",
    category_orders={"label": LABEL_ORDER, "split": ["train", "valid", "test"]},
    title="Distribution des labels par split",
    labels={"count": "Nombre", "label": "Label"},
    color_discrete_sequence=["#3498db", "#f39c12", "#9b59b6"],
)
fig.update_layout(template="plotly_white", height=400)
fig.show()

## 3. Analyse textuelle des declarations

On analyse la longueur des textes (nombre de mots, nombre de caracteres) et leur relation avec les labels.

In [8]:
# Features textuelles de base
df["statement"] = df["statement"].astype(str)
df["n_words"] = df["statement"].str.split().str.len()
df["n_chars"] = df["statement"].str.len()
df["avg_word_len"] = df["n_chars"] / df["n_words"]

print("=== Statistiques textuelles ===")
print(df[["n_words", "n_chars", "avg_word_len"]].describe().round(2))

=== Statistiques textuelles ===
        n_words   n_chars  avg_word_len
count  12791.00  12791.00      12791.00
mean      18.04    107.17          5.99
std       10.13     63.58          0.69
min        2.00     11.00          3.62
25%       12.00     73.00          5.53
50%       17.00     99.00          5.93
75%       22.00    133.00          6.38
max      467.00   3204.00         12.20


In [9]:
# Distribution du nombre de mots par declaration
fig = px.histogram(
    df, x="n_words", color="label_binary_name",
    nbins=60, barmode="overlay", opacity=0.7,
    color_discrete_map={"Fake": "#e74c3c", "Real": "#2ecc71"},
    title="Distribution du nombre de mots par declaration",
    labels={"n_words": "Nombre de mots", "label_binary_name": "Classe"},
)
fig.update_layout(template="plotly_white", height=400)
fig.show()

In [10]:
# Box plot : nombre de mots par label (6 classes)
fig = px.box(
    df, x="label", y="n_words", color="label",
    category_orders={"label": LABEL_ORDER},
    color_discrete_sequence=COLORS_6,
    title="Nombre de mots par classe de veracite",
    labels={"n_words": "Nombre de mots", "label": "Label"},
)
fig.update_layout(showlegend=False, template="plotly_white", height=450)
fig.show()

# Stats par label
print("\nNombre moyen de mots par label :")
print(df.groupby("label")["n_words"].mean().reindex(LABEL_ORDER).round(1))


Nombre moyen de mots par label :
label
pants-fire     17.2
false          16.9
barely-true    18.2
half-true      19.0
mostly-true    18.5
true           18.0
Name: n_words, dtype: float64


In [11]:
# Analyse du vocabulaire : mots les plus frequents par classe (Fake vs Real)
from collections import Counter

def get_top_words(texts, n=20):
    """Retourne les n mots les plus frequents."""
    words = " ".join(texts).lower().split()
    # Retirer stopwords basiques
    stopwords = {"the", "a", "an", "is", "are", "was", "were", "of", "to", "in", 
                 "for", "and", "on", "that", "it", "with", "as", "at", "by", "from",
                 "or", "be", "this", "has", "have", "had", "not", "but", "they",
                 "he", "she", "we", "his", "her", "its", "our", "their", "will",
                 "would", "than", "more", "been", "who", "which", "about", "said"}
    words = [w for w in words if w not in stopwords and len(w) > 2]
    return Counter(words).most_common(n)

top_fake = get_top_words(df[df["label_binary"] == 0]["statement"], 20)
top_real = get_top_words(df[df["label_binary"] == 1]["statement"], 20)

fig = make_subplots(rows=1, cols=2, subplot_titles=["Top mots — Fake", "Top mots — Real"])

fig.add_trace(go.Bar(
    y=[w for w, c in reversed(top_fake)], x=[c for w, c in reversed(top_fake)],
    orientation="h", marker_color="#e74c3c", name="Fake"
), row=1, col=1)

fig.add_trace(go.Bar(
    y=[w for w, c in reversed(top_real)], x=[c for w, c in reversed(top_real)],
    orientation="h", marker_color="#2ecc71", name="Real"
), row=1, col=2)

fig.update_layout(title="Mots les plus frequents par classe", template="plotly_white",
                  height=500, showlegend=False)
fig.show()

## 4. Analyse des metadonnees

Le LIAR dataset fournit des metadonnees riches : **speaker**, **parti politique**, **sujet**, **etat**, **contexte** et l'historique de veracite de chaque speaker (nombre de declarations par categorie).

In [12]:
# Top 15 speakers par nombre de declarations
top_speakers = df["speaker"].value_counts().head(15)

fig = px.bar(
    x=top_speakers.values, y=top_speakers.index,
    orientation="h", color=top_speakers.values,
    color_continuous_scale="Viridis",
    title="Top 15 speakers (nombre de declarations)",
    labels={"x": "Nombre de declarations", "y": "Speaker"},
)
fig.update_layout(template="plotly_white", height=450, coloraxis_showscale=False,
                  yaxis=dict(autorange="reversed"))
fig.show()

print(f"\nNombre total de speakers uniques : {df['speaker'].nunique()}")


Nombre total de speakers uniques : 3309


In [13]:
# Distribution des partis politiques
party_counts = df["party"].value_counts().head(10)

fig = px.bar(
    x=party_counts.index, y=party_counts.values,
    color=party_counts.index,
    color_discrete_map={"republican": "#e74c3c", "democrat": "#3498db", "none": "#95a5a6"},
    title="Distribution par parti politique",
    labels={"x": "Parti", "y": "Nombre de declarations"},
    text=party_counts.values,
)
fig.update_traces(textposition="outside")
fig.update_layout(showlegend=False, template="plotly_white", height=400)
fig.show()

In [14]:
# Veracite par parti (Republicain vs Democrate)
party_df = df[df["party"].isin(["republican", "democrat"])]
party_label = party_df.groupby(["party", "label"]).size().reset_index(name="count")

fig = px.bar(
    party_label, x="label", y="count", color="party",
    barmode="group",
    category_orders={"label": LABEL_ORDER},
    color_discrete_map={"republican": "#e74c3c", "democrat": "#3498db"},
    title="Distribution des labels par parti (Republicain vs Democrate)",
    labels={"count": "Nombre", "label": "Label"},
)
fig.update_layout(template="plotly_white", height=400)
fig.show()

# Taux de Fake par parti
for p in ["republican", "democrat"]:
    sub = df[df["party"] == p]
    fake_rate = (sub["label_binary"] == 0).mean() * 100
    print(f"{p.capitalize()} — Taux de Fake : {fake_rate:.1f}%")

Republican — Taux de Fake : 49.8%
Democrat — Taux de Fake : 33.9%


In [15]:
# Analyse des sujets (subjects)
# Un statement peut avoir plusieurs sujets separes par des virgules
all_subjects = df["subject"].dropna().str.split(",").explode().str.strip()
top_subjects = all_subjects.value_counts().head(15)

fig = px.bar(
    x=top_subjects.values, y=top_subjects.index,
    orientation="h",
    title="Top 15 sujets politiques",
    labels={"x": "Nombre de declarations", "y": "Sujet"},
    color=top_subjects.values, color_continuous_scale="Tealgrn",
)
fig.update_layout(template="plotly_white", height=450, coloraxis_showscale=False,
                  yaxis=dict(autorange="reversed"))
fig.show()

print(f"\nNombre total de sujets uniques : {all_subjects.nunique()}")


Nombre total de sujets uniques : 144


In [16]:
# Credit history des speakers : correlation entre historique et label
history_cols = ["barely_true_counts", "false_counts", "half_true_counts", 
                "mostly_true_counts", "pants_on_fire_counts"]

# Convertir en numerique
for col in history_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

# Score de credibilite : (mostly_true + half_true) / total historique
df["total_history"] = df[history_cols].sum(axis=1)
df["credibility_score"] = np.where(
    df["total_history"] > 0,
    (df["mostly_true_counts"] + df["half_true_counts"]) / df["total_history"],
    0.5  # valeur neutre si pas d'historique
)

fig = px.box(
    df, x="label", y="credibility_score", color="label",
    category_orders={"label": LABEL_ORDER},
    color_discrete_sequence=COLORS_6,
    title="Score de credibilite du speaker par label",
    labels={"credibility_score": "Score de credibilite", "label": "Label"},
)
fig.update_layout(showlegend=False, template="plotly_white", height=450)
fig.show()

## 5. Preprocessing et sauvegarde

On applique deux niveaux de nettoyage :
1. **clean_text** : nettoyage leger (lowercase + ponctuation) — conserve tous les mots
2. **preprocess_nlp** : nettoyage NLP complet (NLTK tokenization + suppression stopwords) — pour TF-IDF/embeddings

**Etapes du preprocessing NLP** :
1. Lowercase
2. Suppression de la ponctuation et des chiffres
3. Tokenisation NLTK
4. Suppression des stopwords anglais
5. Filtrage des tokens courts (< 3 caracteres)

In [17]:
import nltk
nltk.download("punkt_tab", quiet=True)
nltk.download("stopwords", quiet=True)

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

STOP_WORDS = set(stopwords.words("english"))

def clean_text(text: str) -> str:
    """Nettoyage leger : lowercase + suppression ponctuation."""
    text = str(text).lower()
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def preprocess_nlp(text: str) -> str:
    """Preprocessing NLP complet : tokenisation NLTK + suppression stopwords."""
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    tokens = word_tokenize(text)
    tokens = [t for t in tokens if t not in STOP_WORDS and len(t) > 2]
    return " ".join(tokens)

# Appliquer les deux niveaux de nettoyage
df["statement_clean"] = df["statement"].apply(clean_text)
df["statement_nlp"] = df["statement"].apply(preprocess_nlp)

# Verifier
print("Texte original :")
print(f"  {df['statement'].iloc[0]}")
print(f"\nClean (leger) :")
print(f"  {df['statement_clean'].iloc[0]}")
print(f"\nNLP (complet) :")
print(f"  {df['statement_nlp'].iloc[0]}")

# Exemples par classe
print("\n=== Exemples de declarations ===")
for label in LABEL_ORDER:
    sample = df[df["label"] == label]["statement"].iloc[0]
    print(f"\n[{label}] {sample[:120]}...")

Texte original :
  Says the Annies List political group supports third-trimester abortions on demand.

Clean (leger) :
  says the annies list political group supports third trimester abortions on demand

NLP (complet) :
  says annies list political group supports third trimester abortions demand

=== Exemples de declarations ===

[pants-fire] In the case of a catastrophic event, the Atlanta-area offices of the Centers for Disease Control and Prevention will sel...

[false] Says the Annies List political group supports third-trimester abortions on demand....

[barely-true] Jim Dunnam has not lived in the district he represents for years now....

[half-true] When did the decline of coal start? It started when natural gas took off that started to begin in (President George W.) ...

[mostly-true] Hillary Clinton agrees with John McCain "by voting to give George Bush the benefit of the doubt on Iran."...

[true] The Chicago Bears have had more starting quarterbacks in the last 10 years than

In [18]:
# Colonnes a garder pour la modelisation
KEEP_COLS = [
    "id", "label", "label_binary", "label_binary_name",
    "label_3class", "label_3class_name",
    "statement", "statement_clean", "statement_nlp",
    "subject", "speaker", "party", "context",
    "n_words", "n_chars", "credibility_score",
    "split",
]

df_out = df[KEEP_COLS].copy()

# Sauvegarder les splits en Parquet
for split_name in ["train", "valid", "test"]:
    split_df = df_out[df_out["split"] == split_name].reset_index(drop=True)
    out_path = DATA_OUT / f"liar_{split_name}.parquet"
    split_df.to_parquet(out_path, index=False)
    print(f"Sauvegarde {out_path} : {len(split_df):,} lignes")

# Sauvegarder aussi le dataset complet
df_out.to_parquet(DATA_OUT / "liar_complet.parquet", index=False)
print(f"\nDataset complet : {len(df_out):,} lignes -> {DATA_OUT / 'liar_complet.parquet'}")

Sauvegarde ..\data\Traitees\liar_train.parquet : 10,240 lignes
Sauvegarde ..\data\Traitees\liar_valid.parquet : 1,284 lignes
Sauvegarde ..\data\Traitees\liar_test.parquet : 1,267 lignes

Dataset complet : 12,791 lignes -> ..\data\Traitees\liar_complet.parquet


## 6. Synthese de l'EDA

**Observations cles** :

1. **Taille** : ~12 800 declarations, reparties en train (10 240), valid (1 284) et test (1 267)
2. **Desequilibre** : Les 6 classes ne sont pas equilibrees (ratio min/max ~0.39). En binaire, le split est plus equilibre (~50/50)
3. **Texte court** : Median ~17 mots par declaration — un defi pour les modeles NLP
4. **Biais partisan** : Les republicains et democrates dominent le dataset. Le taux de "Fake" varie entre partis — source potentielle de biais
5. **Credit history** : Le score de credibilite du speaker est correle avec le label, ce qui confirme l'utilite des metadonnees
6. **Sujets** : Economie, sante et immigration sont les sujets les plus frequents

**Implications pour la modelisation** :
- Le desequilibre des 6 classes justifie un mapping binaire ou l'utilisation de SMOTE
- Les textes courts favorisent TF-IDF (ngrams) et les modeles contextuels (BERT)
- Attention au **speaker leakage** : le modele ne doit pas simplement memoriser les speakers
- Le vocabulaire politique specifique peut limiter la generalisation out-of-domain